In [2]:
import numpy as np
import pandas as pd

# Пользовательские реализации

## Класс линейной регрессии

In [3]:
import enum


class LinRegMethod(enum.Enum):
    GRADIENT = 0
    SGD = 1
    ANALYTIC = 2


class LinearRegressionCustom:
    def __init__(
        self,
        method=LinRegMethod.ANALYTIC,
        p=2,
        alpha=0.1,
        ratio=None,
        learn_rate=0.01,
        iters=1000,
        batch_size=128,
        random_state=42
    ):
        self.method = method
        self.p = p
        self.alpha = alpha
        self.ratio = ratio
        self.learn_rate = learn_rate
        self.iters = iters
        self.random_state = random_state
        self.batch_size = batch_size
        self.w = None

    def _add_bias(self, X):
        return np.hstack([np.ones((X.shape[0], 1)), X])

    def _reg_grad(self):
        if self.p <= 0 or self.alpha <= 0:
            return 0

        if self.ratio is not None and (self.ratio > 1 or self.ratio < 0):
            raise ValueError('Ratio must be in [0; 1].')

        w = self.w.copy()
        w[0] = 0

        if self.ratio is not None:
            l1_grad = self.ratio * np.sign(w)
            l2_grad = (1 - self.ratio) * w
            return self.alpha * (l1_grad + l2_grad)

        if self.p == 1:
            return self.alpha * np.sign(w)

        if self.p == 2:
            return self.alpha * w

        w_abs = np.abs(w)
        if self.p > 1:
            return self.alpha * (w_abs ** (self.p - 1)) * np.sign(w)

        mask = w_abs > 1e-4
        grad = np.zeros_like(w)
        grad[mask] = self.alpha * w_abs[mask] ** (self.p - 1) * np.sign(w[mask])
        return grad

    def fit(self, X, y):
        X = np.asarray(X, dtype=np.float64)
        y = np.asarray(y, dtype=np.float64).ravel()

        samples, features = X.shape
        X_b = self._add_bias(X)

        if self.method == LinRegMethod.ANALYTIC:
            if self.p <= 0 or self.alpha <= 0:
                self.w = np.linalg.pinv(X_b) @ y
            elif (self.ratio is None and self.p == 2) or self.ratio == 0:
                reg_mtrx = samples * self.alpha * np.eye(features + 1)
                reg_mtrx[0, 0] = 0
                self.w = np.linalg.solve(X_b.T @ X_b + reg_mtrx, X_b.T @ y)
            else:
                raise RuntimeError('Incorrect regularization for analytic method.')
        
        elif self.method == LinRegMethod.GRADIENT:
            self.w = np.zeros(features + 1)
            for _ in range(self.iters):
                dw = (1 / samples) * (X_b.T @ (X_b @ self.w - y)) + self._reg_grad()
                self.w -= self.learn_rate * dw

        elif self.method == LinRegMethod.SGD:
            self.w = np.zeros(features + 1)
            rng = np.random.default_rng(seed=self.random_state)
            for _ in range(self.iters):
                indices = rng.permutation(samples)
                for start in range(0, samples, self.batch_size):
                    batch_idx = indices[start:start+self.batch_size]
                    X_batch = X_b[batch_idx]
                    y_batch = y[batch_idx]
                    dw = (1 / len(batch_idx)) * (X_batch.T @ (X_batch @ self.w - y_batch)) + self._reg_grad()
                    self.w -= self.learn_rate * dw
        else:
            raise ValueError('Unknown training method.')

    def predict(self, X):
        if self.w is None:
            raise RuntimeError('Model should be fitted before predict.')
        X = np.asarray(X, dtype=np.float64)
        X_b = self._add_bias(X)
        return X_b @ self.w

## Нормализация z-score и min-max

In [4]:
class ZScoreScaler:
    def __init__(self, X):
        X = np.asarray(X, dtype=np.float64)
        self.mean = np.mean(X, axis=0)
        self.std = np.std(X, axis=0)
        self.std = np.where(self.std == 0, 1, self.std)

    def scale(self, X):
        X = np.asarray(X, dtype=np.float64)
        return (X - self.mean) / self.std

    def rescale(self, y):
        y = np.asarray(y, dtype=np.float64)
        return y * self.std + self.mean


class MinMaxScaler:
    def __init__(self, X):
        X = np.asarray(X, dtype=np.float64)
        self.min = np.min(X, axis=0)
        x_max = np.max(X, axis=0)
        self.denom = x_max - self.min
        self.denom = np.where(self.denom == 0, 1, self.denom)

    def scale(self, X):
        X = np.asarray(X, dtype=np.float64)
        return (X - self.min) / self.denom

    def rescale(self, y):
        y = np.asarray(y, dtype=np.float64)
        return y * self.denom + self.min

## Метрики MSE, MAE, R^2, MAPE

In [5]:
def validate_arrays(a, b):
    a = np.asarray(a).ravel()
    b = np.asarray(b).ravel()
    if a.shape != b.shape:
        raise ValueError(f'Shapes are not equal: {a.shape} and {b.shape}')
    return a, b


def mean_squared_error_custom(y, a):
    y, a = validate_arrays(y, a)
    return np.mean((y - a) ** 2)


def mean_absolute_error_custom(y, a):
    y, a = validate_arrays(y, a)
    return np.mean(np.abs(y - a))


def r2_score_custom(y, a):
    y, a = validate_arrays(y, a)
    num = np.sum((y - a) ** 2)
    denom = np.sum((y - np.mean(y)) ** 2)
    if np.isclose(denom, 0):
        return 1 if np.isclose(num, 0) else 0
    return 1 - (num / denom)


def mean_absolute_percentage_error_custom(y, a):
    y, a = validate_arrays(y, a)
    return np.mean(np.abs(y - a) / np.maximum(np.abs(y), np.finfo(np.float64).eps))

## Тестирование и сравнение метрик с sklearn

In [6]:
from sklearn.metrics import (
    mean_squared_error,
    mean_absolute_error,
    r2_score,
    mean_absolute_percentage_error,
)

test_arrays = [
    (
        [3.0, 5.0, 2.5, 7.0],
        [2.5, 5.0, 4.0, 8.0],
    ),
    (
        [-1.5, 2.0, -3.5, 4.0],
        [-1.0, 1.5, -4.0, 3.5],
    ),
    (
        [10.0, 20.0, 30.0],
        [10.0, 20.0, 30.0],
    ),
    (
        [5.0, 5.0, 5.0],
        [5.0, 5.0, 5.0],
    ),
    (
        [5.0, 5.0, 5.0],
        [5.0, 5.1, 4.9],
    ),
    (
        [0.0, 2.0, 4.0],
        [0.5, 2.0, 3.5],
    ),
    (
        [[1.0], [2.0], [3.0]],
        [1.1, 1.9, 3.2],
    ),
]

metrics = [
    ('MSE', mean_squared_error_custom, mean_squared_error),
    ('MAE', mean_absolute_error_custom, mean_absolute_error),
    ('R2', r2_score_custom, r2_score),
    ('MAPE', mean_absolute_percentage_error_custom, mean_absolute_percentage_error),
]

for y, a in test_arrays:
    results = []
    for metric_name, custom_fn, sklearn_fn in metrics:
        custom_val = custom_fn(y, a)
        sklearn_val = sklearn_fn(y, a)
        is_equal = np.isclose(custom_val, sklearn_val)
        results.append({
            'Metric': f'{metric_name:<5}',
            'Custom': f'{custom_val:<10.5g}',
            'Sklearn': f'{sklearn_val:<10.5g}',
            'Equal': f'{is_equal}',
        })
    df_results = pd.DataFrame(results)
    print(df_results.to_string(index=False))
    print()

Metric     Custom    Sklearn Equal
 MSE   0.875      0.875       True
 MAE   0.75       0.75        True
 R2    0.72414    0.72414     True
 MAPE  0.22738    0.22738     True

Metric     Custom    Sklearn Equal
 MSE   0.25       0.25        True
 MAE   0.5        0.5         True
 R2    0.9708     0.9708      True
 MAPE  0.2128     0.2128      True

Metric     Custom    Sklearn Equal
 MSE   0          0           True
 MAE   0          0           True
 R2    1          1           True
 MAPE  0          0           True

Metric     Custom    Sklearn Equal
 MSE   0          0           True
 MAE   0          0           True
 R2    1          1           True
 MAPE  0          0           True

Metric     Custom    Sklearn Equal
 MSE   0.0066667  0.0066667   True
 MAE   0.066667   0.066667    True
 R2    0          0           True
 MAPE  0.013333   0.013333    True

Metric     Custom    Sklearn Equal
 MSE   0.16667    0.16667     True
 MAE   0.33333    0.33333     True
 R2    0.9375  

## Сравнение линейной регрессии с sklearn

In [7]:
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression, Ridge, Lasso, ElasticNet, SGDRegressor


train = pd.read_csv('train.csv')
train = train.dropna(ignore_index=True).drop_duplicates(ignore_index=True)
target_outliers_mask = (train.RiskScore < -1000) | (train.RiskScore > 1000)
train = train[~target_outliers_mask]
train = train.select_dtypes(np.number)

y = train['RiskScore']
X = train.drop(columns=['RiskScore'])
X_trainain, X_test, y_trainain, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

X_trainain_scaled = ZScoreScaler(X_trainain).scale(X_trainain)
X_test_scaled = ZScoreScaler(X_test).scale(X_test)

alpha_val = 0.1

tests = [
    {'name': 'Analytic', 'method': LinRegMethod.ANALYTIC, 'p': 0, 'alpha': 0, 'ratio': None},
    {'name': 'Analytic L2', 'method': LinRegMethod.ANALYTIC, 'p': 2, 'alpha': alpha_val, 'ratio': None},

    {'name': 'Gradient', 'method': LinRegMethod.GRADIENT, 'p': 0, 'alpha': 0.0, 'ratio': None},
    {'name': 'Gradient L1', 'method': LinRegMethod.GRADIENT, 'p': 1, 'alpha': alpha_val, 'ratio': None},
    {'name': 'Gradient L2', 'method': LinRegMethod.GRADIENT, 'p': 2, 'alpha': alpha_val, 'ratio': None},
    {'name': 'Gradient ElasticNet 0.5', 'method': LinRegMethod.GRADIENT, 'p': 2, 'alpha': alpha_val, 'ratio': 0.5},
    {'name': 'Gradient Lp 0.5', 'method': LinRegMethod.GRADIENT, 'p': 0.5, 'alpha': alpha_val, 'ratio': None},

    {'name': 'SGD', 'method': LinRegMethod.SGD, 'p': 0, 'alpha': 0, 'ratio': None},
    {'name': 'SGD L1', 'method': LinRegMethod.SGD, 'p': 1, 'alpha': alpha_val, 'ratio': None},
    {'name': 'SGD L2', 'method': LinRegMethod.SGD, 'p': 2, 'alpha': alpha_val, 'ratio': None},
    {'name': 'SGD ElasticNet 0.5', 'method': LinRegMethod.SGD, 'p': 2, 'alpha': alpha_val, 'ratio': 0.5},
]

results = []

for test in tests:
    model = LinearRegressionCustom(
        method=test['method'],
        p=test['p'],
        alpha=test['alpha'],
        ratio=test['ratio'],
        learn_rate=0.05,
        batch_size=128
    )
    model.fit(X_trainain_scaled, y_trainain)    
    preds = model.predict(X_test_scaled)
    results.append({
        'Модель': test['name'],
        'MSE': f'{mean_squared_error(y_test, preds):.2f}',
        'R2': f'{r2_score(y_test, preds):.2f}',
        'Норма весов': f'{np.linalg.norm(model.w[1:]):.2f}',
    })

df_results = pd.DataFrame(results)
print(df_results.to_string(index=False))

results.clear()

sklearn_tests = [
    ('Sklearn LinearRegression', LinearRegression()),
    ('Sklearn L2', Ridge(alpha=alpha_val * len(X_trainain_scaled))),

    ('Sklearn L1', Lasso(alpha=alpha_val)),
    ('Sklearn ElasticNet 0.5', ElasticNet(alpha=alpha_val)),

    ('Sklearn SGD', SGDRegressor(penalty=None, learning_rate='constant', random_state=42)),
    ('Sklearn SGD L1', SGDRegressor(penalty='l1', alpha=alpha_val, learning_rate='constant', random_state=42)),
    ('Sklearn SGD L2', SGDRegressor(penalty='l2', alpha=alpha_val, learning_rate='constant', random_state=42)),
    ('Sklearn SGD ElasticNet 0.5', SGDRegressor(penalty='elasticnet', l1_ratio=0.5, alpha=alpha_val, learning_rate='constant', random_state=42)),
]

for name, model in sklearn_tests:
    model.fit(X_trainain_scaled, y_trainain)
    preds = model.predict(X_test_scaled)
    results.append({
        'Модель': name,
        'MSE': f'{mean_squared_error(y_test, preds):.2f}',
        'R2': f'{r2_score(y_test, preds):.2f}',
        'Норма весов': f'{np.linalg.norm(model.coef_):.2f}',
    })

df_results = pd.DataFrame(results)
print(df_results.to_string(index=False))

                 Модель   MSE   R2 Норма весов
               Analytic 50.13 0.83       10.50
            Analytic L2 51.05 0.83        8.37
               Gradient 50.39 0.83        9.28
            Gradient L1 50.27 0.83        9.19
            Gradient L2 51.05 0.83        8.37
Gradient ElasticNet 0.5 50.69 0.83        8.55
        Gradient Lp 0.5 50.41 0.83        9.54
                    SGD 50.64 0.83       10.43
                 SGD L1 50.55 0.83       10.33
                 SGD L2 51.61 0.83        8.32
     SGD ElasticNet 0.5 51.23 0.83        8.51
                    Модель   MSE   R2 Норма весов
  Sklearn LinearRegression 50.13 0.83       10.50
                Sklearn L2 51.05 0.83        8.37
                Sklearn L1 49.99 0.83       10.67
    Sklearn ElasticNet 0.5 50.68 0.83        8.56
               Sklearn SGD 64.72 0.78       10.43
            Sklearn SGD L1 58.30 0.81        9.78
            Sklearn SGD L2 66.20 0.78        8.38
Sklearn SGD ElasticNet 0.5 56.44 0.8

## Пользовательская кросс-валидация k-fold и leave-one-out

In [8]:
def k_fold_cv(n_samples, n_splits=5, shuffle=True, random_state=42):
    indices = np.arange(n_samples)
    if shuffle:
        rng = np.random.default_rng(random_state)
        rng.shuffle(indices)
    folds = np.array_split(indices, n_splits)
    for i in range(n_splits):
        valid_idx = folds[i]
        train_idx = np.concatenate(folds[:i] + folds[i+1:])
        yield train_idx, valid_idx


def leave_one_out_cv(n_samples):
    indices = np.arange(n_samples)
    for i in range(n_samples):
        valid_idx = indices[i:i+1]
        train_idx = np.concatenate((indices[:i], indices[i+1:]))
        yield train_idx, valid_idx


def cross_val(model, X, y, cv_splits, scoring_func):
    X = np.asarray(X, dtype=np.float64)
    y = np.asarray(y, dtype=np.float64)
    scores = []
    preds = np.empty(y.shape, dtype=y.dtype)
    for train_idx, valid_idx in cv_splits:
        X_train, y_train = X[train_idx], y[train_idx]
        X_valid, y_valid = X[valid_idx], y[valid_idx]

        model.fit(X_train, y_train)
        y_pred = model.predict(X_valid)
        preds[valid_idx] = y_pred

        score = scoring_func(y_valid, y_pred)
        scores.append(score)
    return np.array(scores), preds

In [9]:
k_fold = k_fold_cv(len(X))
cross_val(LinearRegressionCustom(), X, y, k_fold, mean_squared_error)

(array([57.95536275, 61.13707052, 55.86518929, 63.40709683, 57.82054326]),
 array([59.8803707 , 36.29295452, 31.98255976, ..., 47.08067161,
        34.63412855, 51.49315978], shape=(9332,)))

In [10]:
leave_one_out = leave_one_out_cv(len(X))
cross_val(LinearRegressionCustom(), X, y, leave_one_out, mean_squared_error)

(array([38.95828494, 58.90093433,  6.73193355, ..., 34.0749415 ,
        83.1102203 , 21.15033854], shape=(9332,)),
 array([59.93484297, 36.17043152, 31.89350683, ..., 47.05076762,
        34.58320311, 51.5053201 ], shape=(9332,)))